# 2. Data Selection

Following the methodological framework outlined in Step 2 of the *Handbook on Constructing Composite Indicators*, this section details the data selection process, structural scope, and initial quality assessment of the variables chosen for the index. 

To make this research as broad and complete as possible, I set a large scope for the dataset:
* **Timeframe:** I collected data over a 10-year period from **2016 to 2025**. This gives me a decade of data to see how startup environments change over time, capturing major events like the pandemic and the economic recoveries after.
* **Countries:** I included all **217 countries and territories** available in the World Bank database. This allows me to test the index on a truly global scale, comparing high-income nations with developing economies.

## 2.2. Methodological Selection Criteria

In compliance with the OECD guidelines, the 16 variables were selected on the basis of four core analytical parameters:

1. **Analytical Soundness:** Each variable is supported by macroeconomic and entrepreneurial literature, as referenced in Section 1.4.
2. **Measurability:** The framework uses objective, quantitative data rather than survey-based or subjective indicators. The World Bank compiles these metrics from official national sources, including central banks, national statistical offices, and customs authorities.
3. **Country Coverage:** Indicators were selected to ensure broad global coverage, minimising data gaps across the 217 jurisdictions included in the analysis.
4. **Relevance:** Each metric reflects a specific aspect of the startup lifecycle, capturing either market entry conditions or longer-term operational sustainability.

## 2.2. Indicator Types and Direction

To meet the OECD requirement for a data characteristics summary, the table below describes each selected variable by its functional classification (Input or Output) and its direction of impact on the final composite index.

<table>
  <thead>
    <tr>
      <th style="text-align:left">Sub-Index</th>
      <th style="text-align:left">Variable Name</th>
      <th style="text-align:left">Indicator Type</th>
      <th style="text-align:left">Direction</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="4"><b>Finance & Investment</b></td>
      <td>Domestic credit to private sector (% of GDP)</td>
      <td>Input (Capital Availability)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Foreign direct investment, net inflows (% of GDP)</td>
      <td>Output (Market Attraction)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Market capitalization of listed companies (% of GDP)</td>
      <td>Input (Financial Market Size)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Stocks traded, total value (% of GDP)</td>
      <td>Output (Market Liquidity)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td rowspan="4"><b>Tech & Innovation</b></td>
      <td>Fixed broadband subscriptions (per 100 people)</td>
      <td>Input (Physical Connectivity)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Individuals using the Internet (% of population)</td>
      <td>Output (Digital Market Reach)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Secure Internet servers (per 1 million people)</td>
      <td>Input (E-commerce Security)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Research and development (R&D) expenditure (% of GDP)</td>
      <td>Input (Innovation Pipeline)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td rowspan="4"><b>Economic Growth</b></td>
      <td>GDP growth (annual %)</td>
      <td>Output (Economic Momentum)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>Inflation, consumer prices (annual %)</td>
      <td>Process (Stability Indicator)</td>
      <td><b>Negative (-)</b></td>
    </tr>
    <tr>
      <td>Real interest rate (%)</td>
      <td>Process (Cost of Capital)</td>
      <td><b>Negative (-)</b></td>
    </tr>
    <tr>
      <td>Gross fixed capital formation (% of GDP)</td>
      <td>Input (Physical Investment)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td rowspan="4"><b>Talent & Productivity</b></td>
      <td>School enrollment, tertiary (% gross)</td>
      <td>Input (Human Capital Pipeline)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>GDP per person employed (constant 2021 PPP $)</td>
      <td>Output (Labor Productivity)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>High-technology exports (% of manufactured exports)</td>
      <td>Output (Technical Competitiveness)</td>
      <td><b>Positive (+)</b></td>
    </tr>
    <tr>
      <td>ICT goods imports (% total goods imports)</td>
      <td>Input (Supply Chain Integration)</td>
      <td><b>Positive (+)</b></td>
    </tr>
  </tbody>
</table>

## 2.3. Data Strengths and Weaknesses

This section evaluates key strengths and limitations of the dataset in line with OECD guidelines, ensuring transparency before data transformation.

### Strengths

* **Methodological Consistency:** The use of the World Bank DataBank ensures standardised data collection across countries, allowing for reliable cross-country comparisons.
* **Data Integrity:** Indicators are compiled from official sources such as national statistical offices, central banks, and international organisations, ensuring a high level of accuracy and objectivity.

### Limitations

* **Reporting Delays:** Some indicators, such as Research and Development expenditure, are published with a delay, meaning the most recent years may not be fully available.
* **Missing Data:** Certain countries do not report all indicators consistently, which leads to gaps in the dataset.
* **Use of proxies:** Direct global startup-level data is not available across all countries, so broader economic indicators (e.g. financial market activity) are used as substitutes.

### Missing Data
Although the dataset covers 217 states over a 10-year period, reporting gaps remain unavoidable. These limitations justify the need for data imputation techniques, which are addressed in Section 3.

In [1]:
import os
import numpy as np
import pandas as pd

In [7]:
data_path = "../data/worldbank_raw_data.csv"

if os.path.exists(data_path):
    df_raw = pd.read_csv(data_path, na_values=["..", "nan", "NaN"])

    # Clean year column names (e.g. "2016 [YR2016]" -> "2016")
    rename_dict = {}
    for col in df_raw.columns:
        if "[" in col and "YR" in col:
            clean_year = col.split("[")[0].strip()
            rename_dict[col] = clean_year
    df_raw.rename(columns=rename_dict, inplace=True)

    print("Raw World Bank dataset loaded.")
    print(f"Dataset shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns.\n")

    # Map World Bank series codes to readable variable names
    series_mapping = {
        "FS.AST.PRVT.GD.ZS": "credit_to_private_sector",
        "BX.KLT.DINV.WD.GD.ZS": "fdi_inflows",
        "CM.MKT.LCAP.GD.ZS": "market_capitalisation",
        "CM.MKT.TRAD.GD.ZS": "stock_market_liquidity",
        "IT.NET.BBND.P2": "broadband_subscriptions",
        "IT.NET.USER.ZS": "internet_usage_rate",
        "IT.NET.SECR.P6": "secure_servers_density",
        "GB.XPD.RSDV.GD.ZS": "rd_expenditure",
        "NY.GDP.MKTP.KD.ZG": "gdp_growth",
        "FP.CPI.TOTL.ZG": "inflation_rate",
        "FR.INR.RINR": "real_interest_rate",
        "NE.GDI.FTOT.ZS": "capital_formation",
        "SE.TER.ENRR": "tertiary_enrollment",
        "SL.GDP.PCAP.EM.KD": "labor_productivity",
        "TX.VAL.TECH.MF.ZS": "high_tech_exports",
        "TM.VAL.ICTG.ZS.UN": "ict_imports"
    }

    # Keep only selected indicators
    df_filtered = df_raw[df_raw['Series Code'].isin(series_mapping.keys())].copy()
    df_filtered['Variable'] = df_filtered['Series Code'].map(series_mapping)

    # Identify year columns
    year_cols = [col for col in df_filtered.columns if col.isdigit()]

    # Convert year columns to numeric values
    for year in year_cols:
        df_filtered[year] = pd.to_numeric(df_filtered[year], errors='coerce')

    # Missing data summary by variable
    print("----------------- Data Quality Check (Missing Values) -----------------")

    missing_audit = {}
    for var_name, group in df_filtered.groupby('Variable'):
        total_possible = group[year_cols].size
        missing_count = group[year_cols].isnull().sum().sum()
        missing_pct = (missing_count / total_possible) * 100

        missing_audit[var_name] = {
            "Missing Records": missing_count,
            "Missing Percentage (%)": round(missing_pct, 2)
        }

    audit_df = pd.DataFrame(missing_audit).T
    print(audit_df.sort_values(by="Missing Percentage (%)", ascending=False))

else:
    print(f"File not found: {data_path}")

Raw World Bank dataset loaded.
Dataset shape: 3477 rows x 14 columns.

----------------- Data Quality Check (Missing Values) -----------------
                          Missing Records  Missing Percentage (%)
stock_market_liquidity             1530.0                   70.51
market_capitalisation              1506.0                   69.40
rd_expenditure                     1419.0                   65.39
real_interest_rate                 1179.0                   54.33
ict_imports                        1106.0                   50.97
tertiary_enrollment                1016.0                   46.82
credit_to_private_sector            757.0                   34.88
high_tech_exports                   724.0                   33.36
capital_formation                   667.0                   30.74
labor_productivity                  592.0                   27.28
inflation_rate                      547.0                   25.21
internet_usage_rate                 477.0                   21.98

### 2.4. Data Quality Audit Summary

The initial data audit shows clear differences in data completeness across the selected indicators, reflecting variation in global reporting capacity.

* **Financial market coverage gaps:** Capital market indicators show the highest levels of missing data, including `stock_market_liquidity` (70.51%) and `market_capitalisation` (69.40%). This reflects the fact that many countries do not have developed or formally recorded equity markets. 
* **Partial reporting in structural indicators:** Variables such as `rd_expenditure` (65.39%) and `tertiary_enrollment` (46.82%) show moderate to high missingness due to differences in national statistical capacity and reporting practices. 
* **Well-covered core indicators:** Macroeconomic and digital infrastructure variables, including `gdp_growth` (14.01%) and `secure_servers_density` (11.20%), have relatively strong global coverage and form a stable base for the dataset.

These results show that although the dataset is comprehensive, missing values are present across several indicators. Addressing these gaps is necessary before analysis. This leads to Section 3: Data Imputation, where missing data will be handled using appropriate statistical methods